# Fisher combination: contours, addition, marginalisation

This notebook consumes the Fisher `.txt` files produced by the other notebooks
in this folder and demonstrates everything one usually wants to do with them
using only the native CosmicFishPie libraries (`cosmicfishpie.analysis`).

**Where the Fishers live** - `notebooks/results/` (committed to the repo):

| Fisher | Produced by |
|---|---|
| `Euclid_Demo-GCsp-*` (spectro) | `02-Spectro_GCsp-Fisher-nuisance-variants.ipynb` |
| `photometric_pess_vs_opt*` (GCph/WL/GCph-WL) | `03-Photometric_GCph-WL-cross_pess-vs-opt.ipynb` |
| `notebook_cmb_planck_*` (CMB_T/CMB_E) | `04-CMB_Planck.ipynb` (Part 1) |
| `WL_symb_demo_*`, `GCsp_symb_demo_*` | `06-Nautilus_GCsp-WL_2cosmo-nuisance.ipynb` |

**What is shown**

1. all Fishers in one contour plot (`fco.make_triangle_plot`)
2. 1-sigma tables and percentage differences (`CosmicFish_FisherAnalysis`)
3. adding independent-probe Fishers with `+`
4. **marginalise** (Schur complement: integrate other parameters out) vs
   **condition/fix** (drop rows/columns: hold other parameters at their
   fiducial values) - and why the two differ,
5. Fisher-level 'other statistic': Kullback-Leibler **information gain**
   in bits (`fisher_operations.information_gain`).

If `notebooks/results/` contains no Fisher yet, the next cell stops with a
message telling you which notebooks to run first - nothing here overwrites or
recomputes Fishers by itself.

In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from cosmicfishpie.analysis import fisher_matrix as fm_mod
from cosmicfishpie.analysis import fisher_plot_analysis as cfa
from cosmicfishpie.analysis import fisher_operations as cfop
import cosmicfishpie.analysis.fishconsumer as fco

sns.set_theme(context="talk", style="ticks")
snscolors = sns.color_palette("colorblind")

RESULTS = Path("./results")   # notebooks/results/ - committed
PLOTS = Path("./plots")       # notebooks/plots/  - git-ignored
PLOTS.mkdir(parents=True, exist_ok=True)

def label_for(path: Path) -> str:
    name = path.name
    if name.startswith("CosmicFish_"):
        name = name.split("_", 2)[2]      # drop "CosmicFish_<version>_"
    return name[: -len("_FM.txt")] if name.endswith("_FM.txt") else name


fisher_files = sorted(RESULTS.glob("CosmicFish_*_FM.txt"))
if not fisher_files:
    raise RuntimeError(
        f"No Fisher files found in {RESULTS.resolve()}.\n"
        "Run (in this order) and re-run this notebook:\n"
        "  02-Spectro_GCsp-Fisher-nuisance-variants.ipynb   -> spectro Fisher\n"
        "  03-Photometric_GCph-WL-cross_pess-vs-opt.ipynb  -> GCph/WL/cross Fishers\n"
        "  04-CMB_Planck.ipynb                             -> Planck CMB Fisher\n"
        "  06-Nautilus_GCsp-WL_2cosmo-nuisance.ipynb       -> WL/GCsp symbolic Fishers\n"
        "The resulting CosmicFish_*_FM.txt files live in notebooks/results/ and are committed."
    )

fishers = {label_for(p): fm_mod.fisher_matrix(file_name=str(p)) for p in fisher_files}
for lab, f in fishers.items():
    print(f"{lab:55s} {len(f.get_param_names()):3d} params   {f.name}")


## 1. All Fishers in one contour plot

Only parameters present in *every* loaded Fisher can go into a shared triangle
plot, so the parameter list below is the intersection over the loaded set.

In [ ]:
COSMO_ORDER = ["Omegam", "Omegab", "h", "ns", "sigma8", "w0", "wa", "mnu", "Neff"]
all_names = [set(f.get_param_names()) for f in fishers.values()]
common = [p for p in COSMO_ORDER if all(p in s for s in all_names)]
if not common:
    raise RuntimeError("Loaded Fishers share no cosmological parameter - check the files.")

labels = list(fishers)
fisher_list = [fishers[l] for l in labels]

contour = fco.make_triangle_plot(
    fishers=fisher_list,
    fisher_labels=labels,
    params=common,
    colors=snscolors,
    truth_values=fisher_list[0].get_param_names_fiducial_dict(),
    smooth=3,
    bins=30,
    savefile=str(PLOTS / "all_fishers_contours.png"),
)

## 2. 1-sigma table and percentage differences

In [ ]:
def sigma_df(fisher, label, params):
    fid = dict(zip(fisher.get_param_names(), fisher.get_param_fiducial()))
    sig = dict(zip(fisher.get_param_names(), fisher.get_confidence_bounds()))
    rows = []
    for p in params:
        f0, s1 = fid.get(p, np.nan), sig.get(p, np.nan)
        rows.append({"fisher": label, "param": p, "sigma_1s": s1,
                     "percent": 100.0 * s1 / abs(f0) if f0 else np.nan})
    return pd.DataFrame(rows)

sigma_all = pd.concat([sigma_df(f, l, common) for l, f in fishers.items()], ignore_index=True)
display(sigma_all.pivot(index="param", columns="fisher", values="sigma_1s").reindex(common))

first = labels[0]
rel = (sigma_all.pivot(index="param", columns="fisher", values="sigma_1s")
       .div(sigma_all.pivot(index="param", columns="fisher", values="sigma_1s")[first], axis=0)
       .reindex(common))
display((rel - 1.0) * 100.0)  # percent change relative to the first Fisher

## 3. Adding Fisher matrices: `fish_a + fish_b`

`fisher_matrix.__add__` matches parameters **by name**: shared parameters add
their Fisher information (correct for two *independent* probes that both
constrain e.g. `Omegam`), and probe-specific nuisance parameters simply end up
side by side.

Below a spectroscopic and a photometric Fisher are combined when available,
otherwise the first two loaded Fishers are used.

In [ ]:
def _find(*needles):
    for lab in labels:
        low = lab.lower()
        if all(n.lower() in low for n in needles):
            return lab
    return None

lab_gcsp = _find("gcsp") or _find("spectro")
lab_photo = _find("photometric") or _find("photometry")
if lab_gcsp is None or lab_photo is None or lab_gcsp == lab_photo:
    lab_gcsp, lab_photo = labels[0], labels[1]

fish_a, fish_b = fishers[lab_gcsp], fishers[lab_photo]
fish_sum = fish_a + fish_b
fish_sum.name = f"{lab_gcsp} + {lab_photo}"

print("combining:", lab_gcsp, "+", lab_photo)
cmp_list = [fish_a, fish_b, fish_sum]
cmp_labels = [lab_gcsp, lab_photo, "sum"]
cmp_analysis = cfa.CosmicFish_FisherAnalysis(fisher_list=cmp_list)
cmp_analysis.compare_fisher_results()

sigma_cmp = pd.concat([sigma_df(f, l, common) for l, f in zip(cmp_labels, cmp_list)], ignore_index=True)
display(sigma_cmp.pivot(index="param", columns="fisher", values="sigma_1s").reindex(common))

In [ ]:
_ = fco.make_triangle_plot(
    fishers=cmp_list,
    fisher_labels=cmp_labels,
    params=common,
    colors=snscolors[:3],
    truth_values=fish_a.get_param_names_fiducial_dict(),
    smooth=3,
    bins=30,
    shade_fisher=True,
    savefile=str(PLOTS / "fisher_addition.png"),
)

## 4. Marginalising vs. conditioning (fixing)

Two operations that look similar but answer different questions:

| helper | operation | reading of the result |
|---|---|---|
| `fisher_operations.marginalise(F, keep)` / Schur complement over the *complement* of `keep` | integrates the other parameters out | **marginal** errors - what you quote when the others float |
| `fisher_operations.reshuffle(F, keep)` (or `eliminate_parameters` on the *dropped* names) | drops rows/columns | **conditional** errors - what you get when the others are *fixed* at fiducial |

Marginal errors are always $\ge$ conditional ones, sometimes dramatically so
(galaxy bias and IA typically dominate photometric weak lensing).

In [ ]:
# Pick one Fisher (default: the combined one from section 3)
F = fish_sum
keep = [p for p in common if p in F.get_param_names()]

F_marg = cfop.marginalise(F, keep, update_names=False)      # nuisances integrated out
F_cond = cfop.reshuffle(F, keep, update_names=False)        # nuisances held fixed

# both now carry exactly the parameters in `keep`, so sigmas are directly comparable
rows = []
for p in keep:
    i_m = F_marg.get_param_index(p)
    i_c = F_cond.get_param_index(p)
    s_m = F_marg.get_confidence_bounds()[i_m]
    s_c = F_cond.get_confidence_bounds()[i_c]
    rows.append({"param": p, "sigma_marginal": s_m, "sigma_conditional": s_c,
                 "marg/cond": s_m / s_c if s_c else np.nan})
df_mc = pd.DataFrame(rows).set_index("param")
display(df_mc)

In [ ]:
from cosmicfishpie.analysis.fisher_matrix import fisher_matrix as FMcls

def zero_prior_like(f):
    Fz = 0.0 * np.asarray(f.get_fisher_matrix(), dtype=float)
    return FMcls(
        fisher_matrix=Fz,
        param_names=f.get_param_names(),
        param_names_latex=f.get_param_names_latex(),
        fiducial=f.get_param_fiducial(),
    )


# Fisher KL information gain, in bits: how much better fish_a+prior learns
# compared to fish_b+prior (information_gain returns the value; the statistical
# average term is included when stat=True).
info_bits = cfop.information_gain(fish_a, fish_b, zero_prior_like(fish_a), stat=True)
print(f"information gain {lab_gcsp} vs {lab_photo}: {float(info_bits):.4f} bits")

## 5. Recap

* contours: `fco.make_triangle_plot(fishers=..., params=...)`
* percentages / tables: `cfa.CosmicFish_FisherAnalysis.compare_fisher_results()` or the sigma table above
* combining probes: `fish_a + fish_b` (matched by parameter name)
* quote **marginal** errors (`marginalise`) for cosmology with nuisances floating,
  **conditional** ones (`reshuffle`/`eliminate_parameters`) only for fixed-nuisance statements
* KL **information gain** (`fisher_operations.information_gain`) gives the probe difference in bits

Regenerate anything stale by rerunning notebooks 02/03/04/06; the produced
`CosmicFish_*_FM.txt` files in `notebooks/results/` are meant to be committed.